In [ ]:
# Cell 0
# xmltodict 라이브러리 설치 여부를 확인합니다.
!pip list | grep xmltodict

In [ ]:
# Cell 1
# 이 셀은 XML 응답을 딕셔너리로 변환하는 라이브러리를 설치합니다.
!pip install -q xmltodict

In [ ]:
# Cell 2
# 이 셀은 API 요청, XML 변환, 표 정리, 요청 간격에 필요한 라이브러리를 불러옵니다.
import time
import pandas as pd
import requests
import xmltodict

In [ ]:
# Cell 3
# 이 셀은 공공데이터포털에서 발급받은 인증키를 설정합니다.
api_key = "PBJct2NnY43P0PBlqdr3cVao2RJ5bJ0BQOgV9PIOdQFjCCG259NS9FMREH8bJILFOMoxa35pQsOSmLw7Qawprg=="  # 직접 입력

In [ ]:
# Cell 4
# 이 셀은 지역코드와 거래년월을 받아 전체 거래건을 DataFrame으로 반환하는 함수를 만듭니다.
def get_apt_trade_df(lawd_cd, deal_ymd, api_key):
    lawd_cd = str(lawd_cd).strip()
    deal_ymd = str(deal_ymd).strip()

    if not (lawd_cd.isdigit() and len(lawd_cd) == 5):
        raise ValueError("지역코드는 법정동 코드 앞 5자리로 입력해야 합니다.")

    if not (deal_ymd.isdigit() and len(deal_ymd) == 6):
        raise ValueError("거래년월은 YYYYMM 형식의 6자리로 입력해야 합니다.")

    if not api_key or api_key == "여기에_일반_인증키_Decoding_입력":
        raise ValueError("공공데이터포털 인증키를 입력해야 합니다.")

    url = (
        "https://apis.data.go.kr/1613000/"
        "RTMSDataSvcAptTrade/getRTMSDataSvcAptTrade"
    )

    columns = [
        "sggCd",
        "umdNm",
        "aptNm",
        "jibun",
        "excluUseAr",
        "dealYear",
        "dealMonth",
        "dealDay",
        "dealAmount",
        "floor",
        "buildYear",
        "cdealType",
        "cdealDay",
        "dealingGbn",
        "estateAgentSggNm",
        "rgstDate",
        "aptDong",
        "slerGbn",
        "buyerGbn",
        "landLeaseholdGbn",
    ]

    data = []
    page = 1
    num_of_rows = 100

    while True:
        params = {
            "serviceKey": api_key,
            "LAWD_CD": lawd_cd,
            "DEAL_YMD": deal_ymd,
            "pageNo": page,
            "numOfRows": num_of_rows,
        }

        res = requests.get(url, params=params, timeout=10)
        res.raise_for_status()

        result = xmltodict.parse(res.text)
        response = result.get("response")

        if response is None:
            raise ValueError(
                f"예상한 XML 응답 구조가 아닙니다: {res.text[:200]}"
            )

        header = response.get("header", {})
        result_code = str(header.get("resultCode", "")).strip()
        result_msg = str(header.get("resultMsg", "")).strip()

        if result_code == "03":
            return pd.DataFrame(columns=columns)

        if result_code != "000":
            raise ValueError(
                f"API 오류: {result_code} / {result_msg}"
            )

        body = response.get("body", {})
        total_count = int(body.get("totalCount", 0) or 0)
        items = body.get("items")

        if not items:
            break

        page_data = items.get("item", [])

        # 거래건이 한 건이면 딕셔너리로 반환되므로 리스트로 변경합니다.
        if isinstance(page_data, dict):
            page_data = [page_data]

        data.extend(page_data)

        if page * num_of_rows >= total_count:
            break

        page += 1
        time.sleep(0.5)

    if not data:
        return pd.DataFrame(columns=columns)

    df = pd.DataFrame(data)

    # 문자열 앞뒤의 불필요한 공백을 제거합니다.
    for column in df.select_dtypes(include="object").columns:
        df[column] = df[column].map(
            lambda value: value.strip()
            if isinstance(value, str)
            else value
        )

    # 거래금액의 쉼표를 제거하고 숫자로 변환합니다.
    if "dealAmount" in df.columns:
        df["dealAmount"] = pd.to_numeric(
            df["dealAmount"].str.replace(",", "", regex=False),
            errors="coerce",
        ).astype("Int64")

    if "excluUseAr" in df.columns:
        df["excluUseAr"] = pd.to_numeric(
            df["excluUseAr"],
            errors="coerce",
        )

    integer_columns = [
        "dealYear",
        "dealMonth",
        "dealDay",
        "floor",
        "buildYear",
    ]

    for column in integer_columns:
        if column in df.columns:
            df[column] = pd.to_numeric(
                df[column],
                errors="coerce",
            ).astype("Int64")

    # 매뉴얼에 제시된 순서로 열을 정렬합니다.
    ordered_columns = [
        column for column in columns if column in df.columns
    ]
    extra_columns = [
        column for column in df.columns if column not in columns
    ]

    return df[ordered_columns + extra_columns]

In [ ]:
# 법정동 코드 앞 5자리: 중구(26110), 남구(26290), 영도구(26200)
# https://code.go.kr -> 법정동 코드 10자리 중에서 앞 5자리만 가져온 것입니다.
area_code = '26290'

In [ ]:
# 계약년월: YYYYMM 형식
ymonth = '202607'

In [ ]:
# Cell 5
# 이 셀은 지역코드와 거래년월을 입력해 아파트 실거래가를 수집합니다.
df = get_apt_trade_df(
    lawd_cd=area_code,
    deal_ymd=ymonth,
    api_key=api_key,
)

In [ ]:
df.head()

In [ ]:
df.shape

In [ ]:
ymonths = [str(i) for i in range(202601, 202608)]
ymonths

In [ ]:
from tqdm.notebook import tqdm

In [ ]:
result = []
for ymonth in tqdm(ymonths):
    df = get_apt_trade_df(
        lawd_cd=area_code,
        deal_ymd=ymonth,
        api_key=api_key,
    )
    result.append(df)

In [ ]:
apt_df = pd.concat(objs=result, axis=0, ignore_index=True)
apt_df.shape

In [ ]:
apt_df.head()

In [ ]:
apt_df.info()

In [ ]:
apt_df1 = apt_df.loc[:, 'sggCd':'buildYear']
apt_df1.head()

In [ ]:
apt_df1.sort_values(by='dealAmount')

In [ ]:
apt_df1['pricePY'] = apt_df1['dealAmount'] / apt_df1['excluUseAr'] * 3.3
apt_df1.head()

In [ ]:
apt_df1.sort_values(by='pricePY')

In [ ]:
apt_df1.groupby(by='umdNm')['dealAmount'].count().sort_values(ascending=False)

In [ ]:
apt_df1.groupby(by='umdNm')['dealAmount'].mean().sort_values(ascending=False)

In [ ]:
apt_df1.groupby(by='umdNm')['dealAmount'].std().sort_values(ascending=False)

In [ ]:
apt_df1.groupby(by='umdNm')['dealAmount'].min().sort_values(ascending=False)

In [ ]:
apt_df1.groupby(by='umdNm')['dealAmount'].max().sort_values(ascending=False)

In [ ]:
apt_df1.groupby(by='umdNm')['pricePY'].agg(func=['count', 'mean', 'std', 'min', 'max'])

In [ ]:
cond = apt_df1['umdNm'].eq('문현동')
apt_df2 = apt_df1.loc[cond, :]
apt_df2.shape

In [ ]:
imsi = apt_df2.groupby(by='aptNm')['dealAmount'].agg(func=['count', 'mean', 'std', 'min', 'max'])
imsi.sort_values(by='mean', ascending=False)

In [ ]:
imsi = apt_df2.groupby(by='aptNm')['pricePY'].agg(func=['count', 'mean', 'std', 'min', 'max'])
imsi.sort_values(by='mean', ascending=False)

### Colab에서 한글 폰트 설치

In [ ]:
# 현재 Colab에 설치된 폰트 목록을 확인합니다.
!fc-list

In [ ]:
# 불필요한 경로를 제거하고 전체 폰트 이름만 출력합니다.
!fc-list : family

In [ ]:
# Colab에서 사용할 나눔 폰트를 설치합니다.
!apt-get install -y fonts-nanum

In [ ]:
# 불필요한 경로를 제거하고 한글 폰트 이름만 중복 없이 오름차순 정렬하여 출력합니다.
!fc-list :lang=ko family | sort | uniq

### 한글 폰트명 탐색

In [ ]:
# 필요한 모듈을 임포트합니다.
import matplotlib.font_manager as fm
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# 현재 사용 중인 컴퓨터에 설치된 전체 폰트 파일명을 리스트로 생성합니다.
fontList = fm.findSystemFonts(fontext='ttf')

In [ ]:
# fontList의 원소 개수를 확인합니다.
len(fontList)

In [ ]:
# fontList에서 특정 폰트명을 포함하는 파일명을 선택하여 fontPath에 할당합니다.
fontPath = sorted([font for font in fontList if 'Nanum' in font])

In [ ]:
# fontPath를 확인합니다.
fontPath

In [ ]:
# 반복문으로 컴퓨터에 설치된 폰트명을 리스트로 반환합니다.
[fm.FontProperties(fname=font).get_name() for font in fontPath]

In [ ]:
# matplotlib 라이브러리의 폰트 관리자 객체를 초기화합니다.
# [참고] 컴퓨터에 설치된 폰트 파일들을 다시 스캔하여 내부 폰트 캐시를 재구성하여
# 새로 설치한 한글 폰트를 사용할 수 있게 합니다.
fm.fontManager.__init__()

### 그래픽 요소 설정

In [ ]:
# 한글 폰트, 그래프 크기와 해상도 등 그래픽 요소를 설정합니다.
plt.rc(group='font', family='NanumBarunGothic', size=10)
plt.rc(group='figure', figsize=(12, 4), dpi=120)
plt.rc(group='axes', unicode_minus=False)
plt.rc(group='legend', frameon=True, fc='0.9', ec='0.9')

In [ ]:
sns.histplot(data=apt_df, x='dealAmount')
plt.show()

In [ ]:
sns.histplot(data=apt_df1, x='pricePY', binrange=(0, 7000), binwidth=250,
             facecolor='pink', edgecolor='red')
plt.show()

In [ ]:
sns.boxplot(data=apt_df1, y='pricePY')
plt.show()

In [ ]:
sns.countplot(data=apt_df1, x='umdNm')
plt.show()

In [ ]:
sns.barplot(data=apt_df1, x='umdNm', y='pricePY')
plt.show()

In [ ]:
sns.lineplot(data=apt_df1, x='dealMonth', y='dealAmount', hue='umdNm', ci=None)
plt.show()